# Wav2Vec2 Speech Recognition

**Model:** facebook/wav2vec2-base-960h | **Size:** 95MB | **Product:** prod-swnjjrcmk5lzo

Facebook's Wav2Vec2 Base model trained on 960 hours of LibriSpeech data. A self-supervised transformer model fine-tuned for English ASR, optimized for clean speech transcription with competitive accuracy at low model size.

## Use Cases
- English-only speech-to-text pipelines
- Voice assistants and chatbot voice input
- Accessibility tools for real-time captioning
- Audio data labeling and annotation workflows

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'wav2vec2-speech-recognition'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send a base64-encoded English audio payload to the endpoint. The model expects 16kHz mono WAV audio.

In [ ]:
import json
import base64

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Replace with actual base64-encoded 16kHz mono WAV audio bytes
# Example: audio_b64 = base64.b64encode(open('sample_16khz_mono.wav', 'rb').read()).decode('utf-8')
audio_b64 = '<BASE64_ENCODED_16KHZ_MONO_WAV_BYTES>'

payload = json.dumps({'audio': audio_b64})

try:
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=payload
    )
    result_raw = response['Body'].read().decode('utf-8')
    try:
        result = json.loads(result_raw)
        print('Transcription:', result)
    except json.JSONDecodeError:
        print('Raw response:', result_raw)
except Exception as e:
    print(f'Inference failed: {e}')
    raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')